# Sports Betting Arbitrage Scraper (Novibet, Stoiximan, Efbet)
This notebook scrapes odds from three major providers, cleans the data, and identifies arbitrage opportunities.

In [ ]:
import sys
sys.path.append('..')

# Import modules
try:
    from web_scrape_functions import novibet_functions as nv
    from web_scrape_functions import stoiximan_function as stm
    from web_scrape_functions import efbet_function as ef
    import queries as sq
except ImportError as e:
    print(f"Warning: {e}. Ensure web_scrape_functions folder is in the parent directory.")

import pandas as pd
import duckdb
from unidecode import unidecode
from fuzzywuzzy import fuzz
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

In [ ]:
# --- SETUP DRIVER ---
options = webdriver.ChromeOptions()
# options.add_argument("--headless") # Uncomment for background run
options.add_argument("--window-size=1920,1200")
options.add_argument("--disable-blink-features=AutomationControlled")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

## 1. Scraping Data

In [ ]:
# --- 1.1 NOVIBET ---
print("--- Scraping Novibet ---")
try:
    page_url = 'https://www.novibet.gr/en/sports'
    
    # Football
    football_string = nv.novibet_football_text(page_url, driver)
    nv.novibet_football_export(football_string)
    
    # Basketball
    basketball_string = nv.novibet_basketball_text(driver)
    nv.novibet_basketball_export(basketball_string)
    print("Novibet scraping complete.")
    
except Exception as e:
    print(f"Novibet Error: {e}")

# --- 1.2 STOIXIMAN ---
print("\n--- Scraping Stoiximan ---")
try:
    # Football
    football_url = 'https://en.stoiximan.gr/sport/soccer/'
    football_string_stm = stm.stoiximan_football_text(football_url, driver)
    stm.stoiximan_football_export(football_string_stm)

    # Basketball
    basketball_url = 'https://en.stoiximan.gr/sport/basketball/'
    basketball_string_stm = stm.stoiximan_basketball_text(basketball_url, driver)
    stm.stoiximan_basketball_export(basketball_string_stm)
    print("Stoiximan scraping complete.")
    
except Exception as e:
    print(f"Stoiximan Error: {e}")

# --- 1.3 EFBET ---
print("\n--- Scraping Efbet ---")
try:
    # Football
    football_string_ef = ef.efbet_football_text(driver)
    ef.efbet_football_export(football_string_ef)

    # Basketball
    basketball_string_ef = ef.efbet_basketball_text(driver)
    ef.efbet_basketball_export(basketball_string_ef)
    
    # Tennis
    tennis_string_ef = ef.efbet_tennis_text(driver)
    ef.efbet_tennis_export(tennis_string_ef)
    print("Efbet scraping complete.")
    
except Exception as e:
    print(f"Efbet Error: {e}")

driver.quit()

## 2. Data Loading & Cleaning

In [ ]:
def remove_unicode(df):
    return df.apply(lambda x: unidecode(x) if isinstance(x, str) else x)

def clean_df(df):
    if df is None or df.empty: return pd.DataFrame()
    # Standardize columns for queries.py
    col_map = {'One_odd': '1', 'X_odd': 'X', 'Two_odd': '2', 'O_odd': 'O_odds', 'U_odd': 'U_odds'}
    df.rename(columns=col_map, inplace=True)
    
    # Clean text columns
    for col in ['Team1', 'Team2']:
        if col in df.columns:
            df[col] = remove_unicode(df[col].astype(str)).str.lower()
            df[col] = df[col].apply(lambda x: ' '.join([w for w in x.split() if len(w)>2]))
    return df

# Load Data
try:
    df_novi = clean_df(pd.read_csv('data/novibet_football.csv'))
    df_stoi = clean_df(pd.read_csv('data/stoiximan_football.csv'))
    df_efbet = clean_df(pd.read_csv('data/efbet_football.csv'))
except FileNotFoundError:
    print("One or more CSV files are missing. Please run scraping cells first.")
    df_novi, df_stoi, df_efbet = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

## 3. Arbitrage Calculation (SQL)
Runs 3-way arbitrage for Over/Under and 1X2, and 2-way for GG/NG.

In [ ]:
def dbrun(dbcon, query, df1, df2, df3=None):
    dbcon.register('table1', df1) # Novibet
    dbcon.register('table2', df2) # Stoiximan
    if df3 is not None:
        dbcon.register('table3', df3) # Efbet
    return dbcon.query(query).to_df().drop_duplicates()

dbcon = duckdb.connect()

# 1. Over/Under Arbitrage (3-Way)
try:
    df_arb_ou = dbrun(dbcon, sq.query_over_under, df_novi, df_stoi, df_efbet)
    print("--- Over/Under Arbitrage Opportunities (3-Way) ---")
    display(df_arb_ou)
except Exception as e:
    print(f"Over/Under Query Error: {e}")

# 2. 1X2 Arbitrage (3-Way)
try:
    df_arb_1x2 = dbrun(dbcon, sq.query_1X2, df_novi, df_stoi, df_efbet)
    print("\n--- 1X2 Arbitrage Opportunities (3-Way) ---")
    display(df_arb_1x2)
except Exception as e:
    print(f"1X2 Query Error: {e}")

# 3. GG/NG Arbitrage (2-Way: Novibet vs Stoiximan)
try:
    df_arb_ggng = dbrun(dbcon, sq.query_gg_ng, df_novi, df_stoi)
    print("\n--- GG/NG Arbitrage Opportunities (2-Way) ---")
    display(df_arb_ggng)
except Exception as e:
    print(f"GG/NG Query Error: {e}")

## 4. Fuzzy Matching (Advanced)
Finds matches across all 3 providers even if names differ slightly.

In [ ]:
matches = []
print("Running Fuzzy Matching...")

# Iterate through Novibet (Reference)
for index, row in df_novi.iterrows():
    t1_novi = row['Team1']
    t2_novi = row['Team2']
    
    # 1. Find Stoiximan Match
    stoi_match = df_stoi[
        (df_stoi['Team1'].apply(lambda x: fuzz.token_sort_ratio(x, t1_novi)) > 80) &
        (df_stoi['Team2'].apply(lambda x: fuzz.token_sort_ratio(x, t2_novi)) > 80)
    ]
    
    # 2. Find Efbet Match
    ef_match = df_efbet[
        (df_efbet['Team1'].apply(lambda x: fuzz.token_sort_ratio(x, t1_novi)) > 80) &
        (df_efbet['Team2'].apply(lambda x: fuzz.token_sort_ratio(x, t2_novi)) > 80)
    ]
    
    # Proceed if at least one other bookmaker has the match
    if not stoi_match.empty or not ef_match.empty:
        # Get Odds (default to 0.01 to avoid division by zero)
        o_novi = float(row.get('O_odds', 0.01))
        u_novi = float(row.get('U_odds', 0.01))
        
        o_stoi = float(stoi_match.iloc[0]['O_odds']) if not stoi_match.empty else 0.01
        u_stoi = float(stoi_match.iloc[0]['U_odds']) if not stoi_match.empty else 0.01
        
        o_ef = float(ef_match.iloc[0]['O_odds']) if not ef_match.empty else 0.01
        u_ef = float(ef_match.iloc[0]['U_odds']) if not ef_match.empty else 0.01
        
        # Calculate Max Odds
        max_o = max(o_novi, o_stoi, o_ef)
        max_u = max(u_novi, u_stoi, u_ef)
        
        # Basic Arbitrage Check (Odds > 1.0)
        if max_o > 1.0 and max_u > 1.0:
            arb = (1/max_o) + (1/max_u)
            
            if arb < 1.0:
                matches.append({
                    'Match': f"{t1_novi} vs {t2_novi}",
                    'Arb %': round(arb * 100, 2),
                    'ROI %': round((1-arb)*100, 2),
                    'Max Over': max_o,
                    'Max Under': max_u,
                    'O Bookie': 'Novibet' if max_o == o_novi else ('Stoiximan' if max_o == o_stoi else 'Efbet'),
                    'U Bookie': 'Novibet' if max_u == u_novi else ('Stoiximan' if max_u == u_stoi else 'Efbet')
                })

# Create DataFrame
df_fuzzy_arbs = pd.DataFrame(matches).sort_values(by='Arb %')
print(f"Found {len(df_fuzzy_arbs)} fuzzy matches with arbitrage.")
display(df_fuzzy_arbs)